        # Cloud-Native Remote Sensing with STAC, Cloud-Optimized GeoTIFFs and Dask

        **A hands-on tour of analysis-ready geodata that never has to be downloaded.**

        Instead of copying terabytes of imagery to our own disk, we will:

        1. **Find** the right scenes through a **STAC API** (SpatioTemporal Asset Catalog).
        2. **Stream** only the pixels we need from **Cloud-Optimized GeoTIFFs** (COGs).
        3. **Describe** the analysis lazily with **Dask**, then trigger the compute and
           watch the progress bar.

        We will do two real analyses on data hosted in Germany and on Microsoft
        Azure:

        * **Part 1** - 34 years of Landsat: how has the day of peak green-up
          shifted for maize and rapeseed in Lower Saxony?
        * **Part 2** - Can the historical crop type explain the *colour* of the
          bare soil? A national soil composite meets a 10-year crop-type stack.

        > **Audience:** curious scientists. We keep the physics light and the
        > cloud-native mechanics front and centre. All the boilerplate lives in
        > the `eo_utils.py` helper module so we can focus on the ideas.
        

        ## The big idea: the data stays in the cloud

        The classic workflow is *download -> process -> delete*. For a small
        area of interest that is hugely wasteful: a single Landsat scene is
        hundreds of megabytes, and a 30-year time series is many gigabytes,
        even though we may only need a few square kilometres.

        Cloud-native geospatial data flips this around with three building
        blocks:

        | Block | Question it answers | Technology here |
        | --- | --- | --- |
        | **STAC** | *Where is the data?* | `pystac-client`, Planetary Computer & Thünen STAC APIs |
        | **COG** | *How do I read just a part of it?* | HTTP range requests into Cloud-Optimized GeoTIFFs |
        | **Dask** | *How do I compute over more data than fits in RAM?* | lazy, chunked arrays in `xarray` |

        The result: opening a 34-year Landsat stack takes **under a second** and
        uses **zero bytes of RAM** until we actually ask for numbers.

        Let's set up our session as good citizens of the cloud first.
        

In [ ]:
        import warnings

        warnings.filterwarnings("ignore")  # libraries are chatty; keep the demo clean

        import numpy as np
        import pandas as pd

        import eo_utils as eo

        print("Helper module:", eo.__file__)
        

In [ ]:
        # Retries for transient HTTP errors + a limit on simultaneous connections.
        # Cloud providers throttle clients that open hundreds of connections at once.
        eo.apply_cloud_defaults()
        print("Cloud-friendly GDAL/Dask defaults applied.")
        

        ---
        # Part 1 - Three decades of Landsat phenology

        ## 1.1 Connect to the catalogues

        A STAC **client** is just a thin wrapper around a JSON API. We open two:
        Microsoft's **Planetary Computer** (hosting the global Landsat archive)
        and the **Thünen Institute** catalogue (hosting German agricultural
        products). No account or API key is required.
        

In [ ]:
        pc = eo.open_planetary_computer()   # Landsat archive
        thuenen = eo.open_thuenen()         # German agricultural products

        print("Connected to:")
        print(" -", eo.PLANETARY_COMPUTER_URL)
        print(" -", eo.THUENEN_STAC_URL)
        

        ## 1.2 Search for scenes

        We describe what we want with a **bounding box**, a **time range** and a
        **cloud-cover filter**. The search returns lightweight *metadata* items;
        no imagery is transferred. Thünen's `planetary_computer.sign()` step
        below is only needed because Planetary Computer's files are private
        Azure blobs: the helper appends a temporary read token to every URL.
        

In [ ]:
        bbox = [8.98, 53.28, 9.08, 53.32]        # ~7 x 4 km, Lower Saxony
        time_range = "1990-01-01/2023-12-31"     # 34 years

        landsat_items = eo.search_signed_landsat(pc, bbox, time_range, cloud_cover_lt=30)
        print(f"Found {len(landsat_items)} Landsat scenes with < 30% cloud cover")

        example = landsat_items[0]
        print("Example :", example.id)
        print("Date    :", example.datetime.date())
        print("Cloud % :", round(example.properties["eo:cloud_cover"], 1))
        print("Assets  :", ", ".join(list(example.assets)[:8]), "...")
        

        ## 1.3 Load lazily - the heart of the cloud-native idea

        `odc.stac.load` converts the STAC items into an `xarray.Dataset` backed by
        **Dask**. We pass `chunks` to declare the block structure, but *nothing
        is read yet*. The dataset is a promise, not data.

        * `groupby="solar_day"` merges overlapping scenes from the same day.
        * `chunks=(32, 256, 256)` means 32 timesteps x 256 x 256 pixels per block.
        

In [ ]:
        landsat_ds = eo.load_landsat(landsat_items, bbox, bands=("red", "nir08", "qa_pixel"),
                                     chunks=(32, 256, 256))
        landsat_ds
        

In [ ]:
        # A compact look behind the curtain: what is declared, how it is chunked,
        # and how much of it is actually in memory right now.
        eo.dask_report(landsat_ds, "Landsat stack (still lazy)")
        

In [ ]:
        # The dask.array repr exposes the block layout of a single band.
        # Notice: shape is the *whole* array, chunksize is just one block.
        landsat_ds["red"].data
        

        **Pause and notice:** we have 455 timesteps x 149 x 223 pixels declared,
        yet memory use is **0 B**. If we asked for the full data it would be tens
        of megabytes *for this tiny box* - a full Landsat tile is hundreds of
        megabytes per scene, i.e. hundreds of gigabytes over 34 years. Dask lets
        us hold that promise without holding the data.

        ## 1.4 Cloud masking and NDVI (still lazy)

        Before we use the red and near-infrared bands we must

        * scale the digital numbers to surface reflectance
          (`reflectance = DN * 2.75e-5 - 0.2`), and
        * hide cloudy pixels using the `qa_pixel` bit mask (bits for dilated
          cloud, cloud and cloud shadow).

        Every line below simply *extends the Dask graph*. No pixels move yet.
        

In [ ]:
        clear = eo.landsat_clear_mask(landsat_ds)          # True where sky is clear
        ndvi = eo.compute_ndvi(landsat_ds, clear_mask=clear)

        eo.dask_report(ndvi.to_dataset(name="ndvi"), "NDVI (operations chained, still lazy)")
        

        ## 1.5 Day of peak green-up, per year

        For every pixel and every year we want the **day of the year with the
        highest NDVI**. We group the time series by calendar year and ask each
        year for the date of its maximum; fully cloudy pixels are skipped.

        This is the first *real* computation, so we trigger it explicitly and
        watch Dask chew through the blocks.
        

In [ ]:
        max_ndvi_doy = eo.annual_peak_ndvi(ndvi)
        max_ndvi_doy
        

In [ ]:
        # The progress bar is the simplest way to *see* Dask working.
        with eo.compute_with_progress():
            max_ndvi_doy = max_ndvi_doy.compute()

        print("Computed peak-NDVI day for", max_ndvi_doy.sizes["year"], "years")
        

        ## 1.6 Bring in the crop types (aligned to the satellite grid)

        Landsat tells us *when* the peak happened, but not *what* is growing.
        We fetch the Thünen historical crop-type maps for the same years and
        **resample them onto the exact same grid** by passing the Landsat
        geobox. Alignment happens on read; we never store a resampled copy.

        The one class we need first is winter rapeseed (code `1501`) and maize
        (`130`); the full 14-class legend lives in `eo.THUENEN_CLASSES`.
        

In [ ]:
        crop_items = eo.search_crop_types(thuenen, bbox, time_range)
        print("Crop-map years:", len(crop_items))

        crop_ds = eo.load_crop_types(crop_items, geobox=landsat_ds.odc.geobox, chunks=(-1, 256, 256))

        # Make the time axis *annual* so it lines up with our peak-NDVI result.
        crop_ds = crop_ds.assign_coords(year=crop_ds.time.dt.year).swap_dims({"time": "year"}).drop_vars("time")
        crop_ds["crop_type"]
        

        ## 1.7 Median peak day per crop, per year

        For each crop we keep only the pixels where that crop was grown, then
        take the **spatial median** of the peak day for every year. Median is
        robust to the odd stray pixel; `dropna()` removes years with no valid
        observations.
        

In [ ]:
        crop_codes = {"Maize": 130, "Rapeseed": 1501}

        rows = []
        for name, code in crop_codes.items():
            in_crop = crop_ds["crop_type"] == code          # mask: this crop?
            doy = max_ndvi_doy.where(in_crop)               # NaN elsewhere
            median_doy = doy.median(dim=["x", "y"])         # one value per year
            df = median_doy.to_dataframe(name="peak_doy").reset_index()
            df["Crop"] = name
            rows.append(df)

        final_df = pd.concat(rows).dropna()
        final_df.groupby("Crop")["peak_doy"].describe()[["count", "min", "50%", "max"]]
        

In [ ]:
        fig, ax, trends = eo.plot_peak_phenology(
            final_df, savepath="phenology_shift_1990_2023.png"
        )
        print("Linear trend (days per decade):")
        for crop, slope in trends.items():
            print(f"  {crop:10s}: {slope:+.1f}")
        

        **A word of caution.** These trends are a *demonstration*, not an
        authoritative finding. The Landsat constellation changed over time
        (more sensors after 2015 -> more clear observations -> a different
        chance of catching the true peak), and a trend in *detected* peak day
        can partly reflect that changing sampling density. Good practice would
        add uncertainty quantification and a sensitivity test against the
        number of clear-sky observations per pixel.
        

        ### What did Dask actually do?

        * We **declared** a chain of operations: open -> mask -> NDVI -> group by
          year -> pick the maximum. Each step added nodes to a task graph.
        * Only `.compute()` pulled data. Dask read the required **COG windows**
          with HTTP range requests, block by block, fused the operations and
          streamed the result into memory.
        * Memory stayed bounded by the **chunk size**, not the dataset size.

        Want to *watch* it happen? Start a local cluster and open the dashboard
        link it prints:

        ```python
        client = eo.start_dashboard()   # opens http://127.0.0.1:8787/status
        # ... run a .compute() and watch the task stream and memory panel ...
        client.close()
        ```

        (Task-graph images via `dask.visualize()` additionally need the
        `graphviz` binary; the text report from `eo.dask_report()` needs nothing.)

        **Takeaway:** `landsat_ds` was tens of megabytes of "virtual" data and
        **0 B** of real memory. Cloud-native is laziness plus selective reads.
        

        ---
        # Part 2 - Crop types meet the bare-soil composite

        ## 2.1 A different kind of cloud-native data

        Part 1 streamed a *time series* of many small scenes. Part 2 streams two
        very different products:

        * a **categorical annual stack** (the Thünen crop-type maps), and
        * a **single national raster** - a 29 GB bare-soil reflectance composite
          as one Cloud-Optimized GeoTIFF with overviews.

        Question: if we know the *historical* crop at a pixel, does the *colour*
        of the bare soil differ between crops? Soils under maize, rapeseed,
        potato and sugar beet are managed differently, so their reflectance
        might carry a crop-specific fingerprint.

        We work at **100 m** over a ~68 x 78 km loess window - a deliberate
        speed/precision trade-off for a live demo (change `soil_bbox` or the
        resolution to explore).
        

In [ ]:
        soil_bbox = [9.5, 51.9, 10.5, 52.6]      # Braunschweig loess belt
        soil_years = "2013-01-01/2023-12-31"

        crop_items2 = eo.search_crop_types(thuenen, soil_bbox, soil_years)
        print("Crop-map years:", len(crop_items2))

        crop_stack = eo.load_crop_types(crop_items2, bbox=soil_bbox, resolution=100,
                                        chunks=(-1, 256, 256))
        crop_stack
        

        ## 2.2 The majority crop per pixel

        We collapse the 11 annual maps into a single "most frequent crop" layer.
        Two subtleties matter:

        * `0` is the product's nodata value ("not agricultural land"). We turn
          it into `NaN` so it can never win the vote.
        * `xarray.apply_ufunc` with `dask="parallelized"` applies SciPy's `mode`
          **block by block**, with `time` as the reduction dimension. This is how
          you plug an arbitrary NumPy/SciPy function into a lazy array.
        

In [ ]:
        majority_lazy = eo.majority_crop(crop_stack)
        eo.dask_report(majority_lazy, "Majority crop per pixel (lazy)")
        

In [ ]:
        with eo.compute_with_progress():
            majority = majority_lazy.compute()
        majority
        

In [ ]:
        counts = majority.to_series().value_counts()
        summary = pd.DataFrame(
            {"class": [eo.THUENEN_CLASSES.get(int(c), "nodata") for c in counts.index],
             "pixels": counts.values},
            index=counts.index,
        )
        summary
        

In [ ]:
        fig, ax = eo.plot_classified_map(
            majority, title="Historical majority crop type (2013-2023)",
            savepath="majority_crop_map.png",
        )
        

        ## 2.3 Stream the bare-soil composite onto the same grid

        The composite is a single 29 GB COG. We pass the **crop geobox** so
        `odc-stac` reads a matching window and resamples it for us. The helper
        also renames the anonymous `data.1 ... data.10` bands to real names,
        replaces the nodata value and converts the stored integers to
        reflectance.

        COG **overviews** are what make this cheap: for 100 m output, GDAL reads
        a decimated pyramid level instead of the full 10 m data.
        

In [ ]:
        soil_items = eo.search_soil_composite(thuenen, soil_bbox)
        soil_ds = eo.load_soil_composite(soil_items, geobox=crop_stack.odc.geobox)
        eo.dask_report(soil_ds, "Bare-soil composite (lazy)")
        

In [ ]:
        with eo.compute_with_progress():
            soil_ds = soil_ds.compute()
        soil_ds
        

        ## 2.4 Sample the soil spectrum by crop

        For each crop we take the pixels where it is the historical majority,
        drop nodata, trim the extreme 1% of each band (hot pixels and artefacts)
        and stack everything into one tidy table. One row = one bare-soil pixel.
        

In [ ]:
        crop_codes2 = {"Maize": 130, "Rapeseed": 1501, "Potato": 1401, "Sugar Beet": 1402}

        soil_df = eo.build_spectra_dataframe(soil_ds, majority, crop_codes2)
        print(f"{len(soil_df):,} bare-soil pixels across {soil_df['Crop'].nunique()} crops")

        soil_df.groupby("Crop")[["red", "green", "blue"]].mean().round(4)
        

In [ ]:
        fig, ax = eo.plot_mean_spectra(
            soil_df, savepath="bare_soil_spectra_by_crop.png"
        )
        

        ## 2.5 The colour of soil: a chromaticity view

        Finally we reduce each pixel to two intuitive colour coordinates:

        * **x** = red / green (how "red" versus "green" the soil is),
        * **y** = blue / (red + green + blue) (how blue it is).

        Each dot is coloured with the pixel's **true RGB** - we are literally
        plotting the colour of the soil. The ellipses show one standard
        deviation of each crop's distribution; the X marks the mean.
        

In [ ]:
        soil_chroma = eo.add_chromaticity_axes(soil_df)
        fig, ax = eo.plot_chromaticity(
            soil_chroma, savepath="soil_rgb_distribution_zoomed.png"
        )
        

        ### Interpreting (and doubting) the result

        * The four crop distributions overlap heavily - soil colour is driven
          mostly by geology and moisture, not by the last crop. That is a
          realistic, honest result: **crop history explains only part of the
          signal.**
        * The *separation* you do see is a hypothesis, not proof. Confounders
          (soil type, slope, rainfall) are not controlled for here.
        * The majority-crop layer has its own uncertainty: the 1990s and spring
          crops are less reliable, and majority voting erases rotations.

        ---
        ## Wrap-up

        **Cloud-native workflow in four moves**

        1. **Search** metadata with STAC - never download to discover.
        2. **Declare** the computation lazily with Dask - memory stays bounded.
        3. **Compute** on demand - only the touched COG windows are read.
        4. **Align** datasets through geoboxes - resampling happens on read.

        **Try it yourself**

        * Re-run Part 2 for the more reliable period `2017-2023`, or at 30 m.
        * Change `crop_codes` / `crop_codes2` to e.g. `1603` (horticultural
          crops) or `200` (grassland) and see how the spectra change.
        * Add `swir1`/`swir2` to the chromaticity or fit a simple classifier.
        * Start the Dask dashboard (`eo.start_dashboard()`) and watch the graph
          light up during Part 1.

        **References & data**

        * Microsoft Planetary Computer Landsat C2 L2 -
          https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2
        * Thünen Earth Observation (ThEO) STAC - https://eodata.thuenen.de/
        * Tetteh, G. O. et al. (2026). *Nationwide annual agricultural land-use
          maps of Germany from 1990 to 2023 derived from satellite imagery.*
          https://doi.org/10.21203/rs.3.rs-9074257/v1
        * Broeg, T. et al. (2026). *Bare soil reflectance composite for Germany.*
          Remote Sensing of Environment.
        * Frantz, D. (2019). FORCE - Framework for Operational Radiometric
          Correction. https://doi.org/10.3390/rs11091124

        *Data licences: Landsat is public domain; Thünen products are CC-BY 4.0.
        Please cite the providers when you publish.*
        